<a href="https://colab.research.google.com/github/aparnapandey26/Image-Prediction/blob/main/Image_Prediction.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install fastapi uvicorn python-multipart pillow tensorflow nest-asyncio

In [ ]:
import tensorflow as tf
import numpy as np
from tensorflow.keras.applications.mobilenet_v2 import (
    MobileNetV2,
    preprocess_input,
    decode_predictions
)
from tensorflow.keras.preprocessing import image

In [ ]:
model = MobileNetV2(weights="imagenet")
print("Model Loaded Successfully")

14536120/14536120 ━━━━━━━━━━━━━━━━━━━━ 1s 0us/step
Model Loaded Successfully


In [ ]:
from google.colab import files

uploaded = files.upload()


Saving dog.png to dog.png


In [ ]:
import numpy as np
from tensorflow.keras.preprocessing import image
from tensorflow.keras.applications.mobilenet_v2 import (
    preprocess_input,
    decode_predictions
)

# Uploaded image ka filename yahan likho
img_path = "dog.png"      # Agar filename kuch aur hai to yahan change karo

# Load image
img = image.load_img(img_path, target_size=(224, 224))

# Convert to array
img_array = image.img_to_array(img)

# Add batch dimension
img_array = np.expand_dims(img_array, axis=0)

# Preprocess
img_array = preprocess_input(img_array)

# Prediction
predictions = model.predict(img_array)

# Top 3 predictions
results = decode_predictions(predictions, top=3)[0]

print("Prediction Results:")
for _, label, confidence in results:
    print(f"{label}: {confidence*100:.2f}%")

1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 1s/step
35363/35363 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step
Prediction Results:
golden_retriever: 94.02%
kuvasz: 0.20%
Leonberg: 0.18%


In [ ]:
%%writefile app.py

from fastapi import FastAPI, UploadFile, File
from tensorflow.keras.applications.mobilenet_v2 import (
    MobileNetV2,
    preprocess_input,
    decode_predictions
)
from tensorflow.keras.preprocessing import image
from PIL import Image
import numpy as np
import io

app = FastAPI(title="Image Prediction API")

model = MobileNetV2(weights="imagenet")

@app.get("/")
def home():
    return {"message": "Image Prediction API is Running"}

@app.post("/predict")
async def predict(file: UploadFile = File(...)):
    contents = await file.read()

    img = Image.open(io.BytesIO(contents)).convert("RGB")
    img = img.resize((224,224))

    img_array = image.img_to_array(img)
    img_array = np.expand_dims(img_array, axis=0)
    img_array = preprocess_input(img_array)

    prediction = model.predict(img_array)

    result = decode_predictions(prediction, top=1)[0][0]

    return {
        "Prediction": result[1],
        "Confidence": float(result[2] * 100)
    }

Overwriting app.py


In [ ]:
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb
!dpkg -i cloudflared-linux-amd64.deb



Selecting previously unselected package cloudflared.
(Reading database ... 118243 files and directories currently installed.)
Preparing to unpack cloudflared-linux-amd64.deb ...
Unpacking cloudflared (2026.6.1) ...
Setting up cloudflared (2026.6.1) ...
Processing triggers for man-db (2.10.2-1) ...


In [ ]:
import nest_asyncio
import uvicorn
import threading

nest_asyncio.apply()

def run():
    uvicorn.run("app:app", host="0.0.0.0", port=8000)

thread = threading.Thread(target=run)
thread.start()

In [ ]:
!cloudflared tunnel --url http://localhost:8000

2026-07-04T06:48:30Z INF Thank you for trying Cloudflare Tunnel. Doing so, without a Cloudflare account, is a quick way to experiment and try it out. However, be aware that these account-less Tunnels have no uptime guarantee, are subject to the Cloudflare Online Services Terms of Use (https://www.cloudflare.com/website-terms/), and Cloudflare reserves the right to investigate your use of Tunnels for violations of such terms. If you intend to use Tunnels in production you should use a pre-created named tunnel by following: https://developers.cloudflare.com/cloudflare-one/connections/connect-apps
2026-07-04T06:48:30Z INF Requesting new quick Tunnel on trycloudflare.com...
2026-07-04T06:48:36Z INF +--------------------------------------------------------------------------------------------+
2026-07-04T06:48:36Z INF |  Your quick Tunnel has been created! Visit it at (it may take some time to be reachable):  |
2026-07-04T06:48:36Z INF |  https://mrs-heath-machinery-womens.trycloudflare.com 